# MALTO — v6: Sklearn Ensemble (TF-IDF + Stylometric + LightGBM Stacking)

**Goal:** Fast, complementary signal to neural models. Runtime: ~20 min on T4×2.

**Architecture:**
- TF-IDF char (2-6gram) + word (1-3gram) + stylometric features
- Level 1: LinearSVC + LogisticRegression + LightGBM (5-fold OOF each)
- Level 2: LightGBM meta-learner on stacked OOF probs
- No threshold optimization — LightGBM learns the mapping directly


In [ ]:
import os, warnings, gc, time, re
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack as sparse_hstack, csr_matrix
import lightgbm as lgb

warnings.filterwarnings('ignore')
SEED       = 42
NUM_LABELS = 6
LABEL_MAP  = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}
np.random.seed(SEED)

KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input/datasets/aliivaezii/test-and-train',
    '/kaggle/input', '.', '..'
]
DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p; break
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root; break
assert DATA_DIR is not None

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train_df['TEXT'] = train_df['TEXT'].fillna('empty')
test_df['TEXT']  = test_df['TEXT'].fillna('empty')

y_all       = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test  = test_df['TEXT'].values
n_train, n_test = len(texts_train), len(texts_test)

print(f'Train: {n_train}, Test: {n_test}')
print('Class distribution:', dict(zip(*np.unique(y_all, return_counts=True))))
t_start = time.time()


In [ ]:
# ============================================================
# STYLOMETRIC FEATURES
# AI models have characteristic patterns in punctuation density,
# sentence length consistency, vocabulary richness, etc.
# ============================================================
def extract_stylometric(texts):
    features = []
    for text in texts:
        chars  = len(text)
        words  = text.split()
        n_words = len(words)
        # Sentence splitting on ., !, ?
        sents   = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
        n_sents = max(len(sents), 1)

        avg_word_len    = np.mean([len(w) for w in words]) if words else 0
        avg_sent_len    = n_words / n_sents
        sent_lens       = [len(s.split()) for s in sents]
        sent_len_std    = np.std(sent_lens) if len(sent_lens) > 1 else 0
        unique_ratio    = len(set(w.lower() for w in words)) / max(n_words, 1)
        upper_ratio     = sum(c.isupper() for c in text) / max(chars, 1)
        digit_ratio     = sum(c.isdigit() for c in text) / max(chars, 1)

        # Per-char punctuation densities
        def d(s): return text.count(s) / max(chars, 1)
        punct = [
            d(','), d('.'), d('!'), d('?'), d(':'), d(';'),
            (d('(') + d(')')), (d('"') + d("'")),
            (d('-') + d('\u2014')), d('\n'),
        ]

        # Paragraph count
        n_para = max(text.count('\n\n'), 1)

        # Avg words per sentence per paragraph
        words_per_para = n_words / n_para

        features.append([
            np.log1p(chars), np.log1p(n_words), np.log1p(n_sents), np.log1p(n_para),
            avg_word_len, avg_sent_len, sent_len_std, words_per_para,
            unique_ratio, upper_ratio, digit_ratio,
            *punct
        ])
    return np.array(features, dtype=np.float32)

print('Extracting stylometric features...')
X_style_train = extract_stylometric(texts_train)
X_style_test  = extract_stylometric(texts_test)
scaler = StandardScaler()
X_style_train = scaler.fit_transform(X_style_train)
X_style_test  = scaler.transform(X_style_test)
print(f'  Stylometric shape: {X_style_train.shape}')


In [ ]:
# ============================================================
# TF-IDF FEATURES  (char + word n-grams)
# Char n-grams capture punctuation patterns, spacing, AI-specific tokens.
# Wider char range (2-7) than baseline — captures more AI writing signatures.
# ============================================================
print('Building TF-IDF features...')
t0 = time.time()

char_tfidf = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 7),
    max_features=150000, min_df=2, max_df=0.95, sublinear_tf=True)
word_tfidf = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 3),
    max_features=100000, min_df=2, max_df=0.95, sublinear_tf=True)

X_char_train = char_tfidf.fit_transform(texts_train)
X_char_test  = char_tfidf.transform(texts_test)
X_word_train = word_tfidf.fit_transform(texts_train)
X_word_test  = word_tfidf.transform(texts_test)

# Combine all features
X_train = sparse_hstack([X_char_train, X_word_train, csr_matrix(X_style_train)]).tocsr()
X_test  = sparse_hstack([X_char_test,  X_word_test,  csr_matrix(X_style_test)]).tocsr()
print(f'  Total features: {X_train.shape[1]:,}  ({time.time()-t0:.0f}s)')


In [ ]:
# ============================================================
# LEVEL 1: 5-FOLD OOF with LinearSVC + LogisticRegression + LightGBM
# ============================================================
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

l1_oof_svc  = np.zeros((n_train, NUM_LABELS))
l1_oof_lr   = np.zeros((n_train, NUM_LABELS))
l1_oof_lgb  = np.zeros((n_train, NUM_LABELS))
l1_test_svc = np.zeros((n_test,  NUM_LABELS))
l1_test_lr  = np.zeros((n_test,  NUM_LABELS))
l1_test_lgb = np.zeros((n_test,  NUM_LABELS))

fold_f1_svc, fold_f1_lr, fold_f1_lgb = [], [], []

lgb_params = dict(
    n_estimators=2000, learning_rate=0.05, num_leaves=127,
    max_depth=-1, min_child_samples=5,
    subsample=0.8, colsample_bytree=0.5,
    reg_alpha=0.1, reg_lambda=0.1,
    class_weight='balanced', random_state=SEED, n_jobs=-1,
    verbose=-1
)

for fold_idx, (tr, va) in enumerate(skf.split(np.zeros(n_train), y_all)):
    print(f'\nFold {fold_idx+1}/{N_FOLDS}  (tr={len(tr)}, va={len(va)})')
    t_fold = time.time()

    # ── LinearSVC ──────────────────────────────────────────
    svc = CalibratedClassifierCV(
        LinearSVC(C=1.0, class_weight='balanced', max_iter=20000), cv=3, method='sigmoid')
    svc.fit(X_train[tr], y_all[tr])
    l1_oof_svc[va] = svc.predict_proba(X_train[va])
    l1_test_svc   += svc.predict_proba(X_test) / N_FOLDS
    f_svc = f1_score(y_all[va], l1_oof_svc[va].argmax(1), average='macro')
    fold_f1_svc.append(f_svc)
    print(f'  SVC  F1={f_svc:.4f}')

    # ── LogisticRegression ─────────────────────────────────
    lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000,
                            solver='lbfgs', multi_class='multinomial', random_state=SEED)
    lr.fit(X_train[tr], y_all[tr])
    l1_oof_lr[va] = lr.predict_proba(X_train[va])
    l1_test_lr   += lr.predict_proba(X_test) / N_FOLDS
    f_lr = f1_score(y_all[va], l1_oof_lr[va].argmax(1), average='macro')
    fold_f1_lr.append(f_lr)
    print(f'  LR   F1={f_lr:.4f}')

    # ── LightGBM ───────────────────────────────────────────
    lgbm = lgb.LGBMClassifier(**lgb_params)
    lgbm.fit(
        X_train[tr], y_all[tr],
        eval_set=[(X_train[va], y_all[va])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    l1_oof_lgb[va] = lgbm.predict_proba(X_train[va])
    l1_test_lgb   += lgbm.predict_proba(X_test) / N_FOLDS
    f_lgb = f1_score(y_all[va], l1_oof_lgb[va].argmax(1), average='macro')
    fold_f1_lgb.append(f_lgb)
    print(f'  LGB  F1={f_lgb:.4f}  ({time.time()-t_fold:.0f}s)')

print(f'\nLevel-1 OOF Summary:')
print(f'  SVC: {np.mean(fold_f1_svc):.4f} ± {np.std(fold_f1_svc):.4f}')
print(f'  LR:  {np.mean(fold_f1_lr):.4f} ± {np.std(fold_f1_lr):.4f}')
print(f'  LGB: {np.mean(fold_f1_lgb):.4f} ± {np.std(fold_f1_lgb):.4f}')


In [ ]:
# ============================================================
# LEVEL 2: LightGBM meta-learner on stacked OOF predictions
# Input: 18 features (3 models × 6 classes)
# Meta-LR also added for blend diversity
# ============================================================
meta_train = np.hstack([l1_oof_svc, l1_oof_lr, l1_oof_lgb])   # (2400, 18)
meta_test  = np.hstack([l1_test_svc, l1_test_lr, l1_test_lgb]) # (600, 18)

meta_oof_lgb = np.zeros((n_train, NUM_LABELS))
meta_oof_lr  = np.zeros((n_train, NUM_LABELS))
meta_test_lgb_sum = np.zeros((n_test, NUM_LABELS))
meta_test_lr_sum  = np.zeros((n_test, NUM_LABELS))

meta_lgb_params = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=0.5,
    class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1
)

print('Training Level-2 meta-learner...')
for fold_idx, (tr, va) in enumerate(skf.split(np.zeros(n_train), y_all)):
    # LightGBM meta
    meta = lgb.LGBMClassifier(**meta_lgb_params)
    meta.fit(
        meta_train[tr], y_all[tr],
        eval_set=[(meta_train[va], y_all[va])],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    meta_oof_lgb[va] = meta.predict_proba(meta_train[va])
    meta_test_lgb_sum += meta.predict_proba(meta_test) / N_FOLDS

    # LR meta (for blending)
    meta_lr = LogisticRegression(C=5.0, max_iter=1000, random_state=SEED)
    meta_lr.fit(meta_train[tr], y_all[tr])
    meta_oof_lr[va] = meta_lr.predict_proba(meta_train[va])
    meta_test_lr_sum += meta_lr.predict_proba(meta_test) / N_FOLDS

lgb_f1 = f1_score(y_all, meta_oof_lgb.argmax(1), average='macro')
lr_f1  = f1_score(y_all, meta_oof_lr.argmax(1),  average='macro')
print(f'  Meta-LGB OOF F1: {lgb_f1:.4f}')
print(f'  Meta-LR  OOF F1: {lr_f1:.4f}')

# Blend meta predictions (simple average)
final_oof   = 0.6 * meta_oof_lgb + 0.4 * meta_oof_lr
final_test  = 0.6 * meta_test_lgb_sum + 0.4 * meta_test_lr_sum
blend_f1    = f1_score(y_all, final_oof.argmax(1), average='macro')
print(f'  Blend OOF F1:    {blend_f1:.4f}')
print()
print(classification_report(y_all, final_oof.argmax(1),
      target_names=[LABEL_MAP[i] for i in range(NUM_LABELS)]))


In [ ]:
# ============================================================
# SUBMISSION
# ============================================================
final_preds = final_test.argmax(1)

id_col = None
for c in test_df.columns:
    if c.lower() in ('id', 'unnamed: 0', 'index'):
        id_col = c; break

pred_labels = [LABEL_MAP[p] for p in final_preds]
if id_col:
    submission = pd.DataFrame({'id': test_df[id_col], 'LABEL': pred_labels})
else:
    submission = pd.DataFrame({'LABEL': pred_labels})

submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
for i, nm in LABEL_MAP.items():
    cnt = (final_preds == i).sum()
    print(f'  {nm:10s}: {cnt} ({cnt/len(final_preds)*100:.1f}%)')

print(f'\n{"="*50}')
print(f'FINAL SUMMARY (Sklearn Ensemble)')
print(f'  SVC L1 OOF:      {np.mean(fold_f1_svc):.4f}')
print(f'  LR  L1 OOF:      {np.mean(fold_f1_lr):.4f}')
print(f'  LGB L1 OOF:      {np.mean(fold_f1_lgb):.4f}')
print(f'  Meta blend OOF:  {blend_f1:.4f}')
print(f'  Total time:      {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')
print('Submission saved!')
